# `web_search` provider playground

Tests every provider through the unified `web_search()` interface from `tools/web_search.py`.  
**Run the Setup cell first**, then each provider block independently.

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# This notebook lives at mcp/server/tools/ — project root is 3 levels up
ROOT        = Path("__file__").resolve().parents[3]   # HSCodeSearch/
SERVER_ROOT = ROOT / "mcp" / "server"

for p in (str(ROOT), str(SERVER_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(ROOT / ".env")

from tools.web_search import (
    web_search,
    list_providers,
    get_current_config,
    get_available_providers,
)

QUERY = "HS code for Samsung Galaxy S8 smartphone"

def show(result: dict):
    print(f"provider : {result['provider']}")
    print(f"model    : {result.get('model', '')}")
    if result.get("answer"):
        print(f"answer   : {result['answer'][:400]}")
    hits = result.get("search_results", [])
    print(f"results  : {len(hits)}")
    for i, r in enumerate(hits[:5], 1):
        print(f"  [{i}] {r['title']}")
        print(f"      {r['url']}")
        if r.get("snippet"):
            print(f"      {r['snippet'][:110]}")

print("Setup OK  |  all providers:", list_providers())
print("Config    :", get_current_config()["provider"], "/ available:", get_available_providers())

Setup OK  |  all providers: ['brave', 'duckduckgo', 'jina', 'perplexity', 'searxng', 'serper', 'tavily', 'webcrawler']
Config    :  / available: ['duckduckgo', 'searxng', 'tavily', 'webcrawler']


In [2]:
# ── 1. DuckDuckGo (no API key) ─────────────────────────────────────────────
result = web_search(QUERY, provider="duckduckgo")
show(result)

/Users/lscm/Project/PCS/HSCodeSearch/mcp/server/search/providers/duckduckgo.py:36: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS(proxy=self.proxy, timeout=timeout)


provider : duckduckgo
model    : duckduckgo
answer   : ### Search Results for "HS code for Samsung Galaxy S8 smartphone"

*No results found.*
results  : 0


In [3]:
# ── 2. Serper — Google SERP (SERPER_API_KEY) ───────────────────────────────
result = web_search(QUERY, provider="serper", num=5)
show(result)

provider : serper
model    : serper-search
answer   : 

### Search Results for "HS code for Samsung Galaxy S8 smartphone"


**[1] Global Buyers of Samsung Galaxy S8 And Hsn Code 8517 - Volza**
Global Import Trade data of Samsung Galaxy S8 And Hsn Code 8517 ; Date. 27-May-2025 ; HSN Code. 8517792100 ; Product. Description. Samsung Galaxy S8 mobile phone ...

🔗 https://www.volza.com/p/samsung-galaxy-s8/hsn-code-8517/



**[2] Turkey Imports – Samsung g
results  : 5
  [1] Global Buyers of Samsung Galaxy S8 And Hsn Code 8517 - Volza
      https://www.volza.com/p/samsung-galaxy-s8/hsn-code-8517/
      Global Import Trade data of Samsung Galaxy S8 And Hsn Code 8517 ; Date. 27-May-2025 ; HSN Code. 8517792100 ; P
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verif

In [4]:
# ── 3. Tavily AI Search (TAVILY_API_KEY) ───────────────────────────────────
result = web_search(QUERY, provider="tavily", max_results=5)
show(result)

provider : tavily
model    : tavily-basic
answer   : The HS code for Samsung Galaxy S8 smartphone is 85171300. This code applies to smartphones with a specific configuration, as verified by import and export data. The most precise subheading is 8517792100 for the Samsung Galaxy S8 mobile phone.
results  : 5
  [1] Global Buyers of Samsung Galaxy S8 And Hsn Code 8517 - Volza
      https://www.volza.com/p/samsung-galaxy-s8/hsn-code-8517/
      Global Import Trade data of Samsung Galaxy S8 And Hsn Code 8517 ; Date. 27-May-2025 ; HSN Code. 8517792100 ; P
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verified shipment detai
  [3] Global Export Data Under HS Code 8517 - Cybex
      https://www.cybex.in/custom-data/export/all-countries/hs-code-8517
      27-April-

In [4]:
# ── 4. Perplexity AI — LLM answer + citations (PERPLEXITY_API_KEY) ─────────
result = web_search(
    "What is the 6-digit HS code for Samsung Galaxy S8? Give chapter and heading.",
    provider="perplexity",
)
show(result)

provider : perplexity
model    : sonar
answer   : **The 6-digit HS code for the Samsung Galaxy S8 is 851713.**

This code covers **smartphones** for wireless networks, as specified in Chapter **85** (Electrical machinery and equipment), Heading **8517** (Telephone sets, including smartphones and other telephones for cellular networks or for other wireless networks).[1]

Supporting evidence includes:
- Direct classification of smartphones like Sam
results  : 5
  [1] HS Code 85171300 - smartphones - Customs Tariff Number
      https://www.tariffnumber.com/2026/85171300
      HS Code 85171300 - smartphones. Smartphones for wireless networks. German French. warning Can be used for an e
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verified shipment detai
  [3]

In [ ]:
# ── 5. Jina AI Reader (JINA_API_KEY — set key to activate) ─────────────────
if os.getenv("JINA_API_KEY"):
    result = web_search(QUERY, provider="jina", num=5)
    show(result)
else:
    print("⚠️  JINA_API_KEY not set — skipping.")

In [ ]:
# ── 6. Brave Search (BRAVE_API_KEY — set key to activate) ──────────────────
if os.getenv("BRAVE_API_KEY"):
    result = web_search(QUERY, provider="brave", count=5)
    show(result)
else:
    print("⚠️  BRAVE_API_KEY not set — skipping.")

In [ ]:
# ── 7. SearXNG self-hosted (SEARXNG_BASE_URL — set to activate) ────────────
if os.getenv("SEARXNG_BASE_URL"):
    result = web_search(QUERY, provider="searxng")
    show(result)
else:
    print("⚠️  SEARXNG_BASE_URL not set — skipping.")

In [7]:
# ── 8. WebCrawler — crawl a real HS-code lookup page ──────────────────────
import concurrent.futures

URL = "https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA"

# Playwright sync API conflicts with Jupyter's event loop — run in a thread.
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as exe:
    result = exe.submit(web_search, URL, None, False, "webcrawler").result(timeout=60)

# web_search() returns to_dict() — metadata fields are inlined at the top level
print(f"provider : {result['provider']}")
print(f"crawled  : {result.get('crawl_success', result.get('response', {}).get('finish_reason'))}")
print(f"\n── First 800 chars ──")
print(result["answer"][:800])


provider : webcrawler
crawled  : True

── First 800 chars ──
### Search Results for "https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA"

**[1] https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA**
加入收藏 2026年-HSCODE查询   海关HS编码查询系统 申报要素查询 首页 HSCODE目录 热门查询 深入探索 干衣机 洗衣房 家用电器 请输入HS编码或者关键词    您查询的相关hs编码  2 条,您的查询关键词  烘干机 HS编码	品名	实例汇总	申报要素·退税	编码对比 84193990.20	其他烟丝 烘干机 (其他烟丝烘干机) [Tobacco  dryer]	1条	查看详情	-- 84193390.20	冷冻或喷雾式烟丝烘干机 (冷冻或喷雾式烟丝烘干机) [Frozen or spray cut tobacco dryer]	0条	查看详情	对比-84193990.2
*Source: webcrawler*
🔗 [https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA](https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA)

---
*1 results via webcrawler*
